In [ ]:
# Curso: Visión por computadora aplicada a la biodiversidad
# Autor: Hector Chuquillanqui

In [ ]:
import cv2
import os
import time
import numpy as np
import pandas as pd
from google.colab.patches import cv2_imshow
from IPython.display import HTML
# import glob

# Leer imagenes y crear carpeta de salida

In [ ]:
# Funcion par leer archivos
def leer_folder(folder):
  images = []
  filenames = []
  list_files = sorted(os.listdir(folder))
  for filename in list_files:
    img_path = os.path.join(folder, filename)
    f, ext = os.path.splitext(filename)
    img = cv2.imread(img_path)
    if img is not None:
      images.append(img)
      filenames.append(filename)
  return images, filenames

In [ ]:
# Leyendo archivos
images, filenames = leer_folder('/content/originales')
img_dict = dict(zip(filenames, images))
filenames

['imagen01.jpg',
 'imagen02.jpg',
 'imagen03.jpg',
 'imagen04.jpg',
 'imagen05.jpg',
 'imagen06.jpg',
 'imagen07.jpeg',
 'imagen08.jpeg',
 'imagen09.webp',
 'imagen10.jpeg']

In [ ]:
# Crear directorio de salida
directory = "imagenes_segmentadas"

# Revisar existencia de carpeta y crearla.
if not os.path.exists(directory):
    os.makedirs(directory, exist_ok=True)
    print(f"Carpeta '{directory}' creada.")
else:
    print(f"Carpeta '{directory}' ya existe.")

Carpeta 'imagenes_segmentadas' creada.


# Funciones de ayuda

In [ ]:
# Funcion auxiliar: Guardar imagen
def guardar_imagen(filename, img_segemented, type_seg):
  # Obtener el nombre sin extension
  filename_base, ext = os.path.splitext(filename)
  status = cv2.imwrite(os.path.join(directory, filename_base + "_segmented_" + type_seg + ext), img_segemented)
  if status:
    print(f"Imagen {filename} preprocesada guardada.")

# def seleccion_corte(image):
#   # Seleccionar ROI manualmente
#   r = cv2.selectROI("select the area", image)

#   # Cortar imagen segun cuadrante seleccionado
#   cropped_image = image[int(r[1]):int(r[1]+r[3]),
#                         int(r[0]):int(r[0]+r[2])]

#   return cropped_image

# Funcion auxiliar: Convertir a codificacion base64
import base64
# def path_to_base64(image_path, time_processing):
def path_to_base64(image_path_time):
    # image_path_time = image_path_time.split(',')
    image_path = image_path_time[0]
    time_processing = image_path_time[1]
    # Abre la imagen en modo binario y lee su contenido
    with open(image_path, 'rb') as image_file:
        image_data = image_file.read()

    # Codifica la imagen en base64
    encoded_image = base64.b64encode(image_data).decode()

    # Usar imagen en codigo base64 para mostrarla en html
    html_code = f'<p style="text-align:center;">{time_processing}</p> <img src="data:image/jpeg;base64,{encoded_image}" width="300">'

    return(html_code)

## Metodo 1: Gaussian blur + Binning + Thresholding + Extraccion (con mascara)

In [ ]:
def bgremove1(myimage):

    # Adicion de blur Gaussiano, kernel = 5x5
    myimage = cv2.GaussianBlur(myimage,(5,5), 0)

    # Agrupamiento de pixeles (binning) para productir imagenes de 125 colores
    bins=np.array([0,51,102,153,204,255])
    myimage[:,:,:] = np.digitize(myimage[:,:,:],bins,right=True)*51

    # Convertir a escala de grises
    myimage_grey = cv2.cvtColor(myimage, cv2.COLOR_BGR2GRAY)

    # Segmentacion por metodo de Otsu (Otsu thresholding) para el fondo (background).
    # Usando ademas segmentacion binaria para obtener fondo blanco
    ret,background = cv2.threshold(myimage_grey,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)

    # Convertir el background en una imagen de grises
    background = cv2.cvtColor(background, cv2.COLOR_GRAY2BGR)

    # Segmentacion por metodo de Otsu (Otsu thresholding) para el primer plano (background).
    # Usando ademas segmentacion de tipo "TOZERO_INV" para retener algunos detalles
    ret,foreground = cv2.threshold(myimage_grey,0,255,cv2.THRESH_TOZERO_INV+cv2.THRESH_OTSU)  #Currently foreground is only a mask
    foreground = cv2.bitwise_and(myimage,myimage, mask=foreground)  # Update foreground with bitwise_and to extract real foreground

    # Combinar fondo con primer plano
    finalimage = background+foreground

    return finalimage

## Metodo 2: Escala de grises + Thresholding simple + Extraccion + Combinar

In [ ]:
def bgremove2(myimage):
    # Convertir a escala de grises
    myimage_grey = cv2.cvtColor(myimage, cv2.COLOR_BGR2GRAY)

    ret,baseline = cv2.threshold(myimage_grey,127,255,cv2.THRESH_TRUNC)

    ret,background = cv2.threshold(baseline,126,255,cv2.THRESH_BINARY)

    ret,foreground = cv2.threshold(baseline,126,255,cv2.THRESH_BINARY_INV)

    # Interseccion de imagen usando el primer plano binarizado como mascara.
    foreground = cv2.bitwise_and(myimage,myimage, mask=foreground)

    #  Convertir el background en una imagen de grises
    background = cv2.cvtColor(background, cv2.COLOR_GRAY2BGR)

    # Combinar fondo con primer plano
    finalimage = background+foreground

    return finalimage

## Metodo 3: Conversion a HSV + Thresholding + Combinar canales S y V + Extraccion + Combinacion

In [ ]:
def bgremove3(myimage):
    # Convertir imagen RGB (BGR) a HSV
    myimage_hsv = cv2.cvtColor(myimage, cv2.COLOR_BGR2HSV)

    # En el canal S removemos cualquier valor menor a la mitad
    s = myimage_hsv[:,:,1]
    s = np.where(s < 127, 0, 1) # Any value below 127 will be excluded

    # Incremantar el brillo y obtener el modulo de 255 (% 255)
    v = (myimage_hsv[:,:,2] + 127) % 255
    v = np.where(v > 127, 1, 0)  # Any value above 127 will be part of our mask

    # Combinar mascaras S y V en un solo primer plano
    foreground = np.where(s+v > 0, 1, 0).astype(np.uint8)  #Casting back into 8bit integer

    # Tomar el inverso del primer plano para obtener el fondo en 8bits
    background = np.where(foreground==0,255,0).astype(np.uint8)

    # Convertir fondo a BGR
    background = cv2.cvtColor(background, cv2.COLOR_GRAY2BGR)  # Convert background back into BGR space

    # Interseccion de imagen usando el primer plano binarizado como mascara.
    foreground=cv2.bitwise_and(myimage,myimage,mask=foreground)

    # Combinar fondo con primer plano
    finalimage = background+foreground

    return finalimage

## Loop para usar las tres funciones
Obtenemos por cada imagen una lista de 4 elementos (original, metodo1, metodo2, metodo3) y otra lista con los tiempos de ejecucion (en milisegundos) [link text](https:// [link text](https://))

In [ ]:
imgs_list = []
time_list = []
for filename, img_i in img_dict.items():
  original_path = os.path.join("originales", filename)
  img_list_i = [original_path]

  processing_times = []

  def time_mili():
    current = time.time()
    return int(round(current * 1000))

  start_time = time_mili()
  img_bgrem1 = bgremove1(img_i)
  processing_times.append(time_mili() - start_time)

  start_time = time_mili()
  img_bgrem2 = bgremove2(img_i)
  processing_times.append(time_mili() - start_time)

  start_time = time_mili()
  img_bgrem3 = bgremove3(img_i)
  processing_times.append(time_mili() - start_time)

  img_segm = [img_bgrem1, img_bgrem2, img_bgrem3]

  processing_times = [str(x) + ' ms' for x in processing_times]
  processing_times = [None] + processing_times

  filename_base, ext = os.path.splitext(filename)

  for i in range(len(img_segm)):
    segm = 'bgremove' + str(i+1)
    new_path = os.path.join(directory, filename_base + '_segmented_' +  segm + ext)
    img_list_i.append(new_path)
    cv2.imwrite(new_path, img_segm[i])

  imgs_list.append(img_list_i)
  time_list.append(processing_times)

In [ ]:
print(imgs_list)

[['originales/imagen01.jpg', 'imagenes_segmentadas/imagen01_segmented_bgremove1.jpg', 'imagenes_segmentadas/imagen01_segmented_bgremove2.jpg', 'imagenes_segmentadas/imagen01_segmented_bgremove3.jpg'], ['originales/imagen02.jpg', 'imagenes_segmentadas/imagen02_segmented_bgremove1.jpg', 'imagenes_segmentadas/imagen02_segmented_bgremove2.jpg', 'imagenes_segmentadas/imagen02_segmented_bgremove3.jpg'], ['originales/imagen03.jpg', 'imagenes_segmentadas/imagen03_segmented_bgremove1.jpg', 'imagenes_segmentadas/imagen03_segmented_bgremove2.jpg', 'imagenes_segmentadas/imagen03_segmented_bgremove3.jpg'], ['originales/imagen04.jpg', 'imagenes_segmentadas/imagen04_segmented_bgremove1.jpg', 'imagenes_segmentadas/imagen04_segmented_bgremove2.jpg', 'imagenes_segmentadas/imagen04_segmented_bgremove3.jpg'], ['originales/imagen05.jpg', 'imagenes_segmentadas/imagen05_segmented_bgremove1.jpg', 'imagenes_segmentadas/imagen05_segmented_bgremove2.jpg', 'imagenes_segmentadas/imagen05_segmented_bgremove3.jpg'],

In [ ]:
print(time_list)

[[None, '94 ms', '23 ms', '66 ms'], [None, '52 ms', '12 ms', '31 ms'], [None, '12 ms', '3 ms', '9 ms'], [None, '12 ms', '3 ms', '9 ms'], [None, '27 ms', '7 ms', '18 ms'], [None, '18 ms', '5 ms', '14 ms'], [None, '8 ms', '3 ms', '5 ms'], [None, '8 ms', '2 ms', '7 ms'], [None, '17 ms', '5 ms', '19 ms'], [None, '1203 ms', '322 ms', '1019 ms']]


## Generacion de tabla en formato HTML
Creamos las listas de rutas para las imagenes originales y las segmentadas


In [ ]:
segmented_path = '/content/imagenes_segmentadas'
original_path = '/content/originales'
img08_list = sorted(os.listdir(segmented_path))

# Construct full paths for each item
full_paths = []
for item in img08_list:
    full_path = os.path.join(segmented_path, item)
    full_paths.append(full_path)

img08 = [os.path.join(original_path, 'imagen08.jpeg')]
img08_list = img08 + full_paths
print(img08_list)

['/content/originales/imagen08.jpeg', '/content/imagenes_segmentadas/imagen01_segmented_bgremove1.jpg', '/content/imagenes_segmentadas/imagen01_segmented_bgremove2.jpg', '/content/imagenes_segmentadas/imagen01_segmented_bgremove3.jpg', '/content/imagenes_segmentadas/imagen02_segmented_bgremove1.jpg', '/content/imagenes_segmentadas/imagen02_segmented_bgremove2.jpg', '/content/imagenes_segmentadas/imagen02_segmented_bgremove3.jpg', '/content/imagenes_segmentadas/imagen03_segmented_bgremove1.jpg', '/content/imagenes_segmentadas/imagen03_segmented_bgremove2.jpg', '/content/imagenes_segmentadas/imagen03_segmented_bgremove3.jpg', '/content/imagenes_segmentadas/imagen04_segmented_bgremove1.jpg', '/content/imagenes_segmentadas/imagen04_segmented_bgremove2.jpg', '/content/imagenes_segmentadas/imagen04_segmented_bgremove3.jpg', '/content/imagenes_segmentadas/imagen05_segmented_bgremove1.jpg', '/content/imagenes_segmentadas/imagen05_segmented_bgremove2.jpg', '/content/imagenes_segmentadas/imagen0

Agregamos la informacion de las imagenes y el tiempo de ejecucion en una data frame

In [ ]:
col_names = ['Original', 'bgremove1', 'bgremove2', 'bgremove3']
df_imgs = pd.DataFrame(
    [[[item1, item2] for item1, item2 in zip(imgs_list[0], time_list[0])],
     [[item1, item2] for item1, item2 in zip(imgs_list[1], time_list[1])],
     [[item1, item2] for item1, item2 in zip(imgs_list[2], time_list[2])],
     [[item1, item2] for item1, item2 in zip(imgs_list[3], time_list[3])],
     [[item1, item2] for item1, item2 in zip(imgs_list[4], time_list[4])],
     [[item1, item2] for item1, item2 in zip(imgs_list[5], time_list[5])],
     [[item1, item2] for item1, item2 in zip(imgs_list[6], time_list[6])],
     [[item1, item2] for item1, item2 in zip(imgs_list[7], time_list[7])],
     [[item1, item2] for item1, item2 in zip(imgs_list[8], time_list[8])],
     [[item1, item2] for item1, item2 in zip(imgs_list[9], time_list[9])]
     ],
    columns = col_names)

In [ ]:
df_imgs

Aplicamos el formato a las 4 columnas para mostrar las imagenes en formato HTML (requiere conversion de codificacion en base64), junto al tiempo en milisegundos

In [ ]:
formatters_dict = {col: path_to_base64 for col in df_imgs.columns}
display(HTML(df_imgs.to_html(escape=False, formatters=formatters_dict)))

Salvamos la tabla en formato HTML para su posterior descarga

In [ ]:
# Saving the dataframe as a webpage
df_imgs.to_html('webpage.html',escape=False, formatters=formatters_dict)

In [ ]:
!zip -r /content/imagenes_segmentadas.zip /content/imagenes_segmentadas
